<a href="https://colab.research.google.com/github/daniela708/Pirlog_DanielaElena__ActivitatePOO2024/blob/main/VAT_Identifier_Discovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files

uploaded = files.upload()

Saving sample_companies_500.csv to sample_companies_500.csv


In [2]:
import pandas as pd
import numpy as np
import re
import requests

from bs4 import BeautifulSoup

In [3]:
companies = pd.read_csv(
    "sample_companies_500.csv",
    dtype={"CompanyNumber": str}
)

print(companies.shape)

(500, 13)


In [4]:
companies.head()

,CompanyName,CompanyNumber,CompanyCategory,CompanyStatus,RegAddress.PostTown,RegAddress.County,RegAddress.Country,PostCode,SICCode.SicText_1,SICCode.SicText_2,SICCode.SicText_3,SICCode.SicText_4,IncorporationDate
0,ATM POINT LTD,15269778,Private Limited Company,Active,LONDON,NaN,UNITED KINGDOM,N5 2ER,68100 - Buying and selling of own real estate,68201 - Renting and operating of Housing Assoc...,68209 - Other letting and operating of own or ...,68320 - Management of real estate on a fee or ...,08/11/2023
1,A2E VENTURE CATALYSTS LIMITED,10057707,Private Limited Company,Active,MANCHESTER,NaN,ENGLAND,M2 1HW,70221 - Financial management,NaN,NaN,NaN,11/03/2016
2,BEDFORD INSURANCE SERVICES GROUP LIMITED,05333235,Private Limited Company,Active,CHESSINGTON,NaN,ENGLAND,KT9 1BD,65120 - Non-life insurance,NaN,NaN,NaN,14/01/2005
3,24 SEVEN SAMEDAY LTD.,03793935,Private Limited Company,Active,BLABY,LEICESTER,NaN,LE8 4GZ,49410 - Freight transport by road,NaN,NaN,NaN,23/06/1999
4,BRANKSOME LEISURE LIMITED,07499013,Private Limited Company,Active,BOURNEMOUTH,NaN,ENGLAND,BH8 9LZ,68209 - Other letting and operating of own or ...,99999 - Dormant Company,NaN,NaN,19/01/2011


In [5]:
companies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   CompanyName          500 non-null    object
 1   CompanyNumber        500 non-null    object
 2   CompanyCategory      500 non-null    object
 3   CompanyStatus        500 non-null    object
 4   RegAddress.PostTown  497 non-null    object
 5   RegAddress.County    137 non-null    object
 6   RegAddress.Country   442 non-null    object
 7   PostCode             500 non-null    object
 8   SICCode.SicText_1    500 non-null    object
 9   SICCode.SicText_2    117 non-null    object
 10  SICCode.SicText_3    45 non-null     object
 11  SICCode.SicText_4    23 non-null     object
 12  IncorporationDate    500 non-null    object
dtypes: object(13)
memory usage: 50.9+ KB


In [6]:
print("Number of companies:", len(companies))
print("Unique company numbers:", companies["CompanyNumber"].nunique())

Number of companies: 500
Unique company numbers: 500


In [8]:
vat_results = companies.copy()
vat_results.insert(
    0,
    "sample_id",
    range(1, len(vat_results) + 1)
)
vat_results["candidate_vat"] = None
vat_results["source_url"] = None
vat_results["source_type"] = None

vat_results["discovery_status"] = "NOT_SEARCHED"

vat_results["hmrc_valid"] = None
vat_results["hmrc_name"] = None
vat_results["hmrc_address"] = None

vat_results["entity_match"] = None

vat_results["final_status"] = "NOT_SEARCHED"

In [9]:
vat_results[
    [
        "sample_id",
        "CompanyName",
        "CompanyNumber",
        "candidate_vat",
        "discovery_status",
        "final_status"
    ]
].head(10)

,sample_id,CompanyName,CompanyNumber,candidate_vat,discovery_status,final_status
0,1,ATM POINT LTD,15269778,None,NOT_SEARCHED,NOT_SEARCHED
1,2,A2E VENTURE CATALYSTS LIMITED,10057707,None,NOT_SEARCHED,NOT_SEARCHED
2,3,BEDFORD INSURANCE SERVICES GROUP LIMITED,05333235,None,NOT_SEARCHED,NOT_SEARCHED
3,4,24 SEVEN SAMEDAY LTD.,03793935,None,NOT_SEARCHED,NOT_SEARCHED
4,5,BRANKSOME LEISURE LIMITED,07499013,None,NOT_SEARCHED,NOT_SEARCHED
5,6,ASR INVESTMENT SOLUTIONS LTD,16965787,None,NOT_SEARCHED,NOT_SEARCHED
6,7,ABHM LIMITED,11324956,None,NOT_SEARCHED,NOT_SEARCHED
7,8,BARRIERS AND GATES DIRECT LIMITED,16593910,None,NOT_SEARCHED,NOT_SEARCHED
8,9,BRENSCOMBE MANAGEMENT COMPANY LIMITED,02149174,None,NOT_SEARCHED,NOT_SEARCHED
9,10,BRAITHWAITE INDUSTRIES LTD,16450814,None,NOT_SEARCHED,NOT_SEARCHED


In [10]:
vat_results["query_vat"] = (
    '"' + vat_results["CompanyName"] + '" "VAT"'
)

vat_results["query_vat_number"] = (
    '"' + vat_results["CompanyName"] + '" "VAT number"'
)

vat_results["query_vat_registration"] = (
    '"' + vat_results["CompanyName"] + '" "VAT registration"'
)

vat_results["query_company_number"] = (
    '"' +
    vat_results["CompanyName"] +
    '" "' +
    vat_results["CompanyNumber"] +
    '" VAT'
)

vat_results["query_pdf"] = (
    '"' +
    vat_results["CompanyName"] +
    '" VAT filetype:pdf'
)

In [11]:
vat_results[
    [
        "CompanyName",
        "query_vat",
        "query_vat_number",
        "query_company_number"
    ]
].head()

,CompanyName,query_vat,query_vat_number,query_company_number
0,ATM POINT LTD,"""ATM POINT LTD"" ""VAT""","""ATM POINT LTD"" ""VAT number""","""ATM POINT LTD"" ""15269778"" VAT"
1,A2E VENTURE CATALYSTS LIMITED,"""A2E VENTURE CATALYSTS LIMITED"" ""VAT""","""A2E VENTURE CATALYSTS LIMITED"" ""VAT number""","""A2E VENTURE CATALYSTS LIMITED"" ""10057707"" VAT"
2,BEDFORD INSURANCE SERVICES GROUP LIMITED,"""BEDFORD INSURANCE SERVICES GROUP LIMITED"" ""VAT""","""BEDFORD INSURANCE SERVICES GROUP LIMITED"" ""VA...","""BEDFORD INSURANCE SERVICES GROUP LIMITED"" ""05..."
3,24 SEVEN SAMEDAY LTD.,"""24 SEVEN SAMEDAY LTD."" ""VAT""","""24 SEVEN SAMEDAY LTD."" ""VAT number""","""24 SEVEN SAMEDAY LTD."" ""03793935"" VAT"
4,BRANKSOME LEISURE LIMITED,"""BRANKSOME LEISURE LIMITED"" ""VAT""","""BRANKSOME LEISURE LIMITED"" ""VAT number""","""BRANKSOME LEISURE LIMITED"" ""07499013"" VAT"


In [13]:
def extract_gb_vat(text):

    if not isinstance(text, str):
        return []

    pattern = r"\bGB\s*(\d{3})\s*(\d{4})\s*(\d{2})\b"

    matches = re.findall(
        pattern,
        text,
        flags=re.IGNORECASE
    )

    results = []

    for match in matches:
        vat = "GB" + "".join(match)
        results.append(vat)

    return list(dict.fromkeys(results))

In [14]:
test_text = """
ABC Limited
Company Number: 12345678
VAT Registration Number: GB 123 4567 89
London
"""

extract_gb_vat(test_text)

['GB123456789']

In [15]:
def extract_context_vat(text):

    if not isinstance(text, str):
        return []

    pattern = (
        r"(?:VAT\s*(?:Registration\s*)?"
        r"(?:Number|No\.?)?|VAT)"
        r"\s*[:#-]?\s*"
        r"(?:GB\s*)?"
        r"(\d{3})\s*(\d{4})\s*(\d{2})"
    )

    matches = re.findall(
        pattern,
        text,
        flags=re.IGNORECASE
    )

    results = []

    for match in matches:
        vat = "GB" + "".join(match)
        results.append(vat)

    return list(dict.fromkeys(results))

In [16]:
test_text = """
ABC Limited
Company Number: 12345678
Telephone: 123456789

VAT Number: 987 6543 21
"""

extract_context_vat(test_text)

['GB987654321']

In [17]:
def extract_vat_candidates(text):

    candidates = []

    candidates.extend(
        extract_gb_vat(text)
    )

    candidates.extend(
        extract_context_vat(text)
    )

    return list(dict.fromkeys(candidates))

In [18]:
test_page = """
ABC BUILDING SERVICES LTD

Company Number: 12345678

VAT Registration Number:
987 6543 21

Contact:
020 1234 5678

Another reference: GB 111 2222 33
"""

extract_vat_candidates(test_page)

['GB111222233', 'GB987654321']

In [20]:
test10 = vat_results[
    [
        "sample_id",
        "CompanyName",
        "CompanyNumber",
        "PostCode"
    ]
].head(10)

test10

,sample_id,CompanyName,CompanyNumber,PostCode
0,1,ATM POINT LTD,15269778,N5 2ER
1,2,A2E VENTURE CATALYSTS LIMITED,10057707,M2 1HW
2,3,BEDFORD INSURANCE SERVICES GROUP LIMITED,05333235,KT9 1BD
3,4,24 SEVEN SAMEDAY LTD.,03793935,LE8 4GZ
4,5,BRANKSOME LEISURE LIMITED,07499013,BH8 9LZ
5,6,ASR INVESTMENT SOLUTIONS LTD,16965787,UB10 8AD
6,7,ABHM LIMITED,11324956,B97 5ST
7,8,BARRIERS AND GATES DIRECT LIMITED,16593910,CV9 2PD
8,9,BRENSCOMBE MANAGEMENT COMPANY LIMITED,02149174,BH4 8AD
9,10,BRAITHWAITE INDUSTRIES LTD,16450814,NP8 1NY


In [21]:
research_log = test10.copy()

research_log["website_found"] = None
research_log["website_url"] = None
research_log["candidate_vat"] = None
research_log["vat_source_url"] = None
research_log["source_type"] = None
research_log["notes"] = None

research_log

,sample_id,CompanyName,CompanyNumber,PostCode,website_found,website_url,candidate_vat,vat_source_url,source_type,notes
0,1,ATM POINT LTD,15269778,N5 2ER,None,None,None,None,None,None
1,2,A2E VENTURE CATALYSTS LIMITED,10057707,M2 1HW,None,None,None,None,None,None
2,3,BEDFORD INSURANCE SERVICES GROUP LIMITED,05333235,KT9 1BD,None,None,None,None,None,None
3,4,24 SEVEN SAMEDAY LTD.,03793935,LE8 4GZ,None,None,None,None,None,None
4,5,BRANKSOME LEISURE LIMITED,07499013,BH8 9LZ,None,None,None,None,None,None
5,6,ASR INVESTMENT SOLUTIONS LTD,16965787,UB10 8AD,None,None,None,None,None,None
6,7,ABHM LIMITED,11324956,B97 5ST,None,None,None,None,None,None
7,8,BARRIERS AND GATES DIRECT LIMITED,16593910,CV9 2PD,None,None,None,None,None,None
8,9,BRENSCOMBE MANAGEMENT COMPANY LIMITED,02149174,BH4 8AD,None,None,None,None,None,None
9,10,BRAITHWAITE INDUSTRIES LTD,16450814,NP8 1NY,None,None,None,None,None,None


In [22]:
test10 = vat_results[
    ["sample_id", "CompanyName", "CompanyNumber", "PostCode"]
].head(10)

test10

,sample_id,CompanyName,CompanyNumber,PostCode
0,1,ATM POINT LTD,15269778,N5 2ER
1,2,A2E VENTURE CATALYSTS LIMITED,10057707,M2 1HW
2,3,BEDFORD INSURANCE SERVICES GROUP LIMITED,05333235,KT9 1BD
3,4,24 SEVEN SAMEDAY LTD.,03793935,LE8 4GZ
4,5,BRANKSOME LEISURE LIMITED,07499013,BH8 9LZ
5,6,ASR INVESTMENT SOLUTIONS LTD,16965787,UB10 8AD
6,7,ABHM LIMITED,11324956,B97 5ST
7,8,BARRIERS AND GATES DIRECT LIMITED,16593910,CV9 2PD
8,9,BRENSCOMBE MANAGEMENT COMPANY LIMITED,02149174,BH4 8AD
9,10,BRAITHWAITE INDUSTRIES LTD,16450814,NP8 1NY


In [23]:
pilot = vat_results.head(10).copy()

pilot["search_status"] = "NOT_SEARCHED"
pilot["candidate_vat"] = None
pilot["candidate_source"] = None
pilot["source_type"] = None
pilot["hmrc_status"] = "NOT_CHECKED"
pilot["hmrc_name"] = None
pilot["hmrc_address"] = None
pilot["entity_match"] = None
pilot["notes"] = None

In [24]:
mask = pilot["CompanyNumber"] == "03793935"

pilot.loc[mask, "search_status"] = "CANDIDATE_FOUND"
pilot.loc[mask, "candidate_vat"] = "GB770300368"
pilot.loc[mask, "candidate_source"] = (
    "https://vat-lookup.co.uk/verify/"
    "vat_check.php/VATNumber/GB770300368"
)
pilot.loc[mask, "source_type"] = "THIRD_PARTY_VAT_AGGREGATOR"
pilot.loc[mask, "notes"] = (
    "Candidate only. Third-party source reports same "
    "company name and registered address. Requires HMRC verification."
)

In [25]:
pilot[
    [
        "CompanyName",
        "CompanyNumber",
        "candidate_vat",
        "search_status",
        "hmrc_status"
    ]
]

,CompanyName,CompanyNumber,candidate_vat,search_status,hmrc_status
0,ATM POINT LTD,15269778,None,NOT_SEARCHED,NOT_CHECKED
1,A2E VENTURE CATALYSTS LIMITED,10057707,None,NOT_SEARCHED,NOT_CHECKED
2,BEDFORD INSURANCE SERVICES GROUP LIMITED,05333235,None,NOT_SEARCHED,NOT_CHECKED
3,24 SEVEN SAMEDAY LTD.,03793935,GB770300368,CANDIDATE_FOUND,NOT_CHECKED
4,BRANKSOME LEISURE LIMITED,07499013,None,NOT_SEARCHED,NOT_CHECKED
5,ASR INVESTMENT SOLUTIONS LTD,16965787,None,NOT_SEARCHED,NOT_CHECKED
6,ABHM LIMITED,11324956,None,NOT_SEARCHED,NOT_CHECKED
7,BARRIERS AND GATES DIRECT LIMITED,16593910,None,NOT_SEARCHED,NOT_CHECKED
8,BRENSCOMBE MANAGEMENT COMPANY LIMITED,02149174,None,NOT_SEARCHED,NOT_CHECKED
9,BRAITHWAITE INDUSTRIES LTD,16450814,None,NOT_SEARCHED,NOT_CHECKED


In [26]:
mask = pilot["CompanyNumber"] == "03793935"

pilot.loc[mask, "hmrc_status"] = "VALID"
pilot.loc[mask, "hmrc_name"] = "24 SEVEN SAMEDAY LTD"
pilot.loc[mask, "hmrc_address"] = (
    "UNIT 1A, WINCHESTER AVENUE, "
    "BLABY INDUSTRIAL PARK, BLABY, "
    "LEICESTER, LE8 4GZ, GB"
)

pilot.loc[mask, "entity_match"] = "MATCH"

pilot.loc[mask, "final_status"] = "VERIFIED"

In [27]:
pilot.loc[
    pilot["CompanyNumber"] == "03793935",
    [
        "CompanyName",
        "CompanyNumber",
        "PostCode",
        "candidate_vat",
        "hmrc_status",
        "hmrc_name",
        "entity_match",
        "final_status"
    ]
]

,CompanyName,CompanyNumber,PostCode,candidate_vat,hmrc_status,hmrc_name,entity_match,final_status
3,24 SEVEN SAMEDAY LTD.,03793935,LE8 4GZ,GB770300368,VALID,24 SEVEN SAMEDAY LTD,MATCH,VERIFIED


In [19]:
extract_vat_candidates(test_page)

['GB111222233', 'GB987654321']

In [28]:
n_pilot = len(pilot)

n_candidates = (
    pilot["candidate_vat"]
    .notna()
    .sum()
)

n_valid = (
    pilot["hmrc_status"]
    .eq("VALID")
    .sum()
)

n_verified = (
    pilot["final_status"]
    .eq("VERIFIED")
    .sum()
)

n_wrong_entity = (
    pilot["entity_match"]
    .eq("MISMATCH")
    .sum()
)

print("Companies searched:", n_pilot)
print("Candidates found:", n_candidates)
print("HMRC valid:", n_valid)
print("Verified company-VAT links:", n_verified)
print("Wrong entity matches:", n_wrong_entity)

Companies searched: 10
Candidates found: 1
HMRC valid: 1
Verified company-VAT links: 1
Wrong entity matches: 0


In [29]:
if n_candidates > 0:
    candidate_precision = n_verified / n_candidates
    false_positive_rate = (
        n_candidates - n_verified
    ) / n_candidates

    print(
        "Candidate precision:",
        round(candidate_precision * 100, 2),
        "%"
    )

    print(
        "False-positive rate:",
        round(false_positive_rate * 100, 2),
        "%"
    )

Candidate precision: 100.0 %
False-positive rate: 0.0 %


In [30]:
no_candidate_numbers = [
    "15269778",  # ATM POINT LTD
    "10057707",  # A2E VENTURE CATALYSTS LIMITED
    "05333235",  # BEDFORD INSURANCE SERVICES GROUP LIMITED
    "07499013",  # BRANKSOME LEISURE LIMITED
    "16965787",  # ASR INVESTMENT SOLUTIONS LTD
    "11324956",  # ABHM LIMITED
    "16593910",  # BARRIERS AND GATES DIRECT LIMITED
    "02149174",  # BRENSCOMBE MANAGEMENT COMPANY LIMITED
    "16450814"   # BRAITHWAITE INDUSTRIES LTD
]

mask = pilot["CompanyNumber"].isin(no_candidate_numbers)

pilot.loc[mask, "search_status"] = "NO_CANDIDATE_INITIAL_SEARCH"

pilot.loc[mask, "notes"] = (
    "No VAT candidate identified in the initial "
    "exact-name + company-number web search. "
    "This does not imply that the company is not VAT registered."
)

In [31]:
pilot[
    [
        "CompanyName",
        "CompanyNumber",
        "candidate_vat",
        "search_status",
        "hmrc_status",
        "final_status"
    ]
]

,CompanyName,CompanyNumber,candidate_vat,search_status,hmrc_status,final_status
0,ATM POINT LTD,15269778,None,NO_CANDIDATE_INITIAL_SEARCH,NOT_CHECKED,NOT_SEARCHED
1,A2E VENTURE CATALYSTS LIMITED,10057707,None,NO_CANDIDATE_INITIAL_SEARCH,NOT_CHECKED,NOT_SEARCHED
2,BEDFORD INSURANCE SERVICES GROUP LIMITED,05333235,None,NO_CANDIDATE_INITIAL_SEARCH,NOT_CHECKED,NOT_SEARCHED
3,24 SEVEN SAMEDAY LTD.,03793935,GB770300368,CANDIDATE_FOUND,VALID,VERIFIED
4,BRANKSOME LEISURE LIMITED,07499013,None,NO_CANDIDATE_INITIAL_SEARCH,NOT_CHECKED,NOT_SEARCHED
5,ASR INVESTMENT SOLUTIONS LTD,16965787,None,NO_CANDIDATE_INITIAL_SEARCH,NOT_CHECKED,NOT_SEARCHED
6,ABHM LIMITED,11324956,None,NO_CANDIDATE_INITIAL_SEARCH,NOT_CHECKED,NOT_SEARCHED
7,BARRIERS AND GATES DIRECT LIMITED,16593910,None,NO_CANDIDATE_INITIAL_SEARCH,NOT_CHECKED,NOT_SEARCHED
8,BRENSCOMBE MANAGEMENT COMPANY LIMITED,02149174,None,NO_CANDIDATE_INITIAL_SEARCH,NOT_CHECKED,NOT_SEARCHED
9,BRAITHWAITE INDUSTRIES LTD,16450814,None,NO_CANDIDATE_INITIAL_SEARCH,NOT_CHECKED,NOT_SEARCHED


In [32]:
strategy2 = pilot[
    pilot["final_status"] == "NO_CANDIDATE_YET"
][[
    "sample_id",
    "CompanyName",
    "CompanyNumber",
    "PostCode"
]].copy()

strategy2["website_url"] = None
strategy2["page_url"] = None
strategy2["candidate_vat"] = None
strategy2["source_type"] = None
strategy2["extraction_status"] = "NOT_CHECKED"
strategy2["notes"] = None

strategy2

,sample_id,CompanyName,CompanyNumber,PostCode,website_url,page_url,candidate_vat,source_type,extraction_status,notes


In [33]:
def get_page_text(url):
    headers = {
        "User-Agent": "Mozilla/5.0 VAT-research-proof-of-concept"
    }

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=15
        )

        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        for tag in soup(["script", "style", "noscript"]):
            tag.decompose()

        text = soup.get_text(" ", strip=True)

        return text

    except Exception as e:
        print(f"Could not access {url}: {e}")
        return None

In [34]:
def find_vat_on_page(url):
    text = get_page_text(url)

    if text is None:
        return []

    return extract_vat_candidates(text)

In [35]:
pilot["final_status"].value_counts(dropna=False)

,count
final_status,
NOT_SEARCHED,9
VERIFIED,1


In [36]:
mask = pilot["search_status"] == "NO_CANDIDATE_INITIAL_SEARCH"

pilot.loc[mask, "final_status"] = "NO_CANDIDATE_YET"

In [37]:
pilot["final_status"].value_counts(dropna=False)

,count
final_status,
NO_CANDIDATE_YET,9
VERIFIED,1


In [38]:
strategy2 = pilot[
    pilot["final_status"] == "NO_CANDIDATE_YET"
][[
    "sample_id",
    "CompanyName",
    "CompanyNumber",
    "PostCode"
]].copy()

strategy2["website_url"] = None
strategy2["page_url"] = None
strategy2["candidate_vat"] = None
strategy2["source_type"] = None
strategy2["extraction_status"] = "NOT_CHECKED"
strategy2["notes"] = None

strategy2

,sample_id,CompanyName,CompanyNumber,PostCode,website_url,page_url,candidate_vat,source_type,extraction_status,notes
0,1,ATM POINT LTD,15269778,N5 2ER,None,None,None,None,NOT_CHECKED,None
1,2,A2E VENTURE CATALYSTS LIMITED,10057707,M2 1HW,None,None,None,None,NOT_CHECKED,None
2,3,BEDFORD INSURANCE SERVICES GROUP LIMITED,05333235,KT9 1BD,None,None,None,None,NOT_CHECKED,None
4,5,BRANKSOME LEISURE LIMITED,07499013,BH8 9LZ,None,None,None,None,NOT_CHECKED,None
5,6,ASR INVESTMENT SOLUTIONS LTD,16965787,UB10 8AD,None,None,None,None,NOT_CHECKED,None
6,7,ABHM LIMITED,11324956,B97 5ST,None,None,None,None,NOT_CHECKED,None
7,8,BARRIERS AND GATES DIRECT LIMITED,16593910,CV9 2PD,None,None,None,None,NOT_CHECKED,None
8,9,BRENSCOMBE MANAGEMENT COMPANY LIMITED,02149174,BH4 8AD,None,None,None,None,NOT_CHECKED,None
9,10,BRAITHWAITE INDUSTRIES LTD,16450814,NP8 1NY,None,None,None,None,NOT_CHECKED,None


In [39]:
mask = strategy2["CompanyNumber"] == "05333235"

strategy2.loc[mask, "website_url"] = "https://www.bedfordinsurance.co.uk/"
strategy2.loc[mask, "source_type"] = "OFFICIAL_WEBSITE"

In [40]:
url = "https://www.bedfordinsurance.co.uk/"

text = get_page_text(url)

if text:
    print("Page successfully downloaded.")
    print("Characters extracted:", len(text))
    print(text[:1000])
else:
    print("Page could not be downloaded.")

Page successfully downloaded.
Characters extracted: 2453
Bedford Insurance Group Skip to content Search: Search Monday - Friday: 9am - 6:00pm  |  Saturday: 9am - 4:30pm  |  Sunday: Closed Bedford Insurance Your UK Based Insurance Broker Home Our Brands Insurance Services Commercial Insurance Services Landlord Insurance About Us Careers Contact Us Complaints Claims Hotline Home Our Brands Insurance Services Commercial Insurance Services About Us Careers Contact Us Complaints Terms & Conditions Claims Hotline We protect the things you love. Founded in 1965, with the aim to help find our customers the right cover at a competitive price. Our Brands. The last two years has seen significant change to our business including a rebrand of our customer facing businesses. New and existing customers can contact us by following the links below. Car, van or home.
Get the right insurance. Helping you get a great deal on your van, car andhome insurance. Get A Quote No matter how big or small,
your bus

In [41]:
candidates = find_vat_on_page(url)

print("VAT candidates found:", candidates)

VAT candidates found: []


In [42]:
from urllib.parse import urljoin, urlparse

def find_relevant_links(base_url):
    headers = {
        "User-Agent": "Mozilla/5.0 VAT-research-proof-of-concept"
    }

    keywords = [
        "terms",
        "conditions",
        "legal",
        "privacy",
        "contact",
        "about"
    ]

    try:
        response = requests.get(
            base_url,
            headers=headers,
            timeout=15
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        relevant_links = []

        for link in soup.find_all("a", href=True):
            href = link["href"]
            anchor_text = link.get_text(" ", strip=True)

            full_url = urljoin(base_url, href)

            # păstrăm doar linkurile din același domeniu
            if urlparse(full_url).netloc != urlparse(base_url).netloc:
                continue

            combined_text = (
                anchor_text + " " + full_url
            ).lower()

            if any(keyword in combined_text for keyword in keywords):
                relevant_links.append(full_url)

        return list(dict.fromkeys(relevant_links))

    except Exception as e:
        print("Error:", e)
        return []

In [43]:
relevant_links = find_relevant_links(url)

print("Relevant pages found:")

for link in relevant_links:
    print(link)

Relevant pages found:
https://www.bedfordinsurance.co.uk/about-us/
https://www.bedfordinsurance.co.uk/contact-us/
https://www.bedfordinsurance.co.uk/contact-us/#claims-hotline
https://www.bedfordinsurance.co.uk/terms-conditions/
https://www.bedfordinsurance.co.uk/wp-content/uploads/2026/03/Bedford-Insurance-Privacy-Policy-v17032026-1.pdf


In [44]:
results = []

for page in relevant_links:

    candidates = find_vat_on_page(page)

    results.append({
        "page_url": page,
        "vat_candidates": candidates
    })

results

[{'page_url': 'https://www.bedfordinsurance.co.uk/about-us/',
  'vat_candidates': []},
 {'page_url': 'https://www.bedfordinsurance.co.uk/contact-us/',
  'vat_candidates': []},
 {'page_url': 'https://www.bedfordinsurance.co.uk/contact-us/#claims-hotline',
  'vat_candidates': []},
 {'page_url': 'https://www.bedfordinsurance.co.uk/terms-conditions/',
  'vat_candidates': []},
 {'page_url': 'https://www.bedfordinsurance.co.uk/wp-content/uploads/2026/03/Bedford-Insurance-Privacy-Policy-v17032026-1.pdf',
  'vat_candidates': []}]

In [45]:
final_results = companies.copy()

final_results["candidate_vat"] = None
final_results["candidate_source"] = None
final_results["source_type"] = None

final_results["hmrc_status"] = "NOT_CHECKED"
final_results["hmrc_name"] = None
final_results["hmrc_address"] = None

final_results["entity_match"] = "NOT_CHECKED"
final_results["final_status"] = "NO_CANDIDATE"

print("Number of companies:", len(final_results))

final_results.head()

Number of companies: 500


,CompanyName,CompanyNumber,CompanyCategory,CompanyStatus,RegAddress.PostTown,RegAddress.County,RegAddress.Country,PostCode,SICCode.SicText_1,SICCode.SicText_2,...,SICCode.SicText_4,IncorporationDate,candidate_vat,candidate_source,source_type,hmrc_status,hmrc_name,hmrc_address,entity_match,final_status
0,ATM POINT LTD,15269778,Private Limited Company,Active,LONDON,NaN,UNITED KINGDOM,N5 2ER,68100 - Buying and selling of own real estate,68201 - Renting and operating of Housing Assoc...,...,68320 - Management of real estate on a fee or ...,08/11/2023,None,None,None,NOT_CHECKED,None,None,NOT_CHECKED,NO_CANDIDATE
1,A2E VENTURE CATALYSTS LIMITED,10057707,Private Limited Company,Active,MANCHESTER,NaN,ENGLAND,M2 1HW,70221 - Financial management,NaN,...,NaN,11/03/2016,None,None,None,NOT_CHECKED,None,None,NOT_CHECKED,NO_CANDIDATE
2,BEDFORD INSURANCE SERVICES GROUP LIMITED,05333235,Private Limited Company,Active,CHESSINGTON,NaN,ENGLAND,KT9 1BD,65120 - Non-life insurance,NaN,...,NaN,14/01/2005,None,None,None,NOT_CHECKED,None,None,NOT_CHECKED,NO_CANDIDATE
3,24 SEVEN SAMEDAY LTD.,03793935,Private Limited Company,Active,BLABY,LEICESTER,NaN,LE8 4GZ,49410 - Freight transport by road,NaN,...,NaN,23/06/1999,None,None,None,NOT_CHECKED,None,None,NOT_CHECKED,NO_CANDIDATE
4,BRANKSOME LEISURE LIMITED,07499013,Private Limited Company,Active,BOURNEMOUTH,NaN,ENGLAND,BH8 9LZ,68209 - Other letting and operating of own or ...,99999 - Dormant Company,...,NaN,19/01/2011,None,None,None,NOT_CHECKED,None,None,NOT_CHECKED,NO_CANDIDATE


In [46]:
mask = final_results["CompanyNumber"] == "03793935"

final_results.loc[mask, "candidate_vat"] = "GB770300368"
final_results.loc[mask, "candidate_source"] = "VAT directory"
final_results.loc[mask, "source_type"] = "THIRD_PARTY_VAT_AGGREGATOR"

final_results.loc[mask, "hmrc_status"] = "VALID"
final_results.loc[mask, "hmrc_name"] = "24 SEVEN SAMEDAY LTD"
final_results.loc[mask, "hmrc_address"] = (
    "UNIT 1A, WINCHESTER AVENUE, "
    "BLABY INDUSTRIAL PARK, BLABY, "
    "LEICESTER, LE8 4GZ, GB"
)

final_results.loc[mask, "entity_match"] = "MATCH"
final_results.loc[mask, "final_status"] = "VERIFIED"

In [47]:
final_results[
    final_results["CompanyNumber"] == "03793935"
][[
    "CompanyName",
    "CompanyNumber",
    "candidate_vat",
    "hmrc_status",
    "entity_match",
    "final_status"
]]

,CompanyName,CompanyNumber,candidate_vat,hmrc_status,entity_match,final_status
3,24 SEVEN SAMEDAY LTD.,03793935,GB770300368,VALID,MATCH,VERIFIED


In [48]:
final_results.loc[
    final_results["CompanyNumber"] != "03793935",
    "final_status"
] = "NOT_PROCESSED"

In [49]:
final_results["final_status"].value_counts()

,count
final_status,
NOT_PROCESSED,499
VERIFIED,1


In [50]:
SERPER_API_KEY = "b581895b3a7cd2bd72173b3e40ae003f472be98e"

In [51]:
import requests

def serper_search(query):
    url = "https://google.serper.dev/search"

    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json"
    }

    payload = {
        "q": query,
        "gl": "uk",
        "hl": "en",
        "num": 10
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=20
    )

    response.raise_for_status()

    return response.json()

In [52]:
query = '"24 SEVEN SAMEDAY LTD" "03793935" VAT'

test_search = serper_search(query)

print("Query:", query)

for result in test_search.get("organic", []):
    print("\nTITLE:", result.get("title"))
    print("LINK:", result.get("link"))
    print("SNIPPET:", result.get("snippet"))

Query: "24 SEVEN SAMEDAY LTD" "03793935" VAT

TITLE: Icom - (225)05815552/03793935
LINK: https://www.facebook.com/100064268636976/photos/2250581555203793935/443442744474676/
SNIPPET: Facebook · disponible chez ICOM. cel:05815552/03793935 · L9 plus disponible chez ICOM. cel: 05815552/03793935 · tout achat a ICOM vous donnes droit ...

TITLE: Commande ton smartphone Evertek ici avec un an... ...
LINK: https://www.facebook.com/100064268636976/videos/commande-ton-smartphone-evertek-ici-avec-un-an-0581555203793935/1108531762600110/
SNIPPET: Infinix merci: 05815552/03793935 ... EVERTEK disponible chez Icom 05815552/03793935 achetez smart. 00:31. EVERTEK disponible chez Icom 05815552/ ...

TITLE: David Michael HANCOCK personal appointments
LINK: https://find-and-update.company-information.service.gov.uk/officers/_G532sdQFka0cCUm8kchTq_aIkc/appointments
SNIPPET: 24 SEVEN SAMEDAY LTD. (03793935). Company status: Active. Correspondence address: Unit 1a Winchester Avenue, Blaby Ind Park, Blaby, L

In [53]:
def discover_vat_candidates(company_name, company_number):

    queries = [
        f'"{company_name}" "VAT number"',
        f'"{company_name}" "VAT registration"',
        f'"{company_name}" VAT',
        f'"{company_name}" "{company_number}" VAT'
    ]

    candidates = []

    for query in queries:
        try:
            data = serper_search(query)

            for result in data.get("organic", []):
                title = result.get("title", "")
                snippet = result.get("snippet", "")
                link = result.get("link", "")

                search_text = title + " " + snippet

                vats = extract_vat_candidates(search_text)

                for vat in vats:
                    candidates.append({
                        "candidate_vat": vat,
                        "source_url": link,
                        "query": query
                    })

        except Exception as e:
            print("Search error:", company_name, e)

    # eliminăm duplicatele
    unique = {}

    for item in candidates:
        vat = item["candidate_vat"]

        if vat not in unique:
            unique[vat] = item

    return list(unique.values())

In [54]:
test_candidates = discover_vat_candidates(
    "24 SEVEN SAMEDAY LTD",
    "03793935"
)

test_candidates

[{'candidate_vat': 'GB770300368',
  'source_url': 'http://vat-lookup.com/verify/vat_check.php/VATNumber/GB770300368',
  'query': '"24 SEVEN SAMEDAY LTD" "VAT number"'},
 {'candidate_vat': 'GB496785415',
  'source_url': 'https://www.formationdata.co.uk/director/david-michael-hancock-_G532sdQFka0cCUm8kchTq_aIkc',
  'query': '"24 SEVEN SAMEDAY LTD" "VAT number"'},
 {'candidate_vat': 'GB740669225',
  'source_url': 'http://www.vat-lookup.co.uk/verify/vat_check.php/VATNumber/GB740669225',
  'query': '"24 SEVEN SAMEDAY LTD" "VAT number"'},
 {'candidate_vat': 'GB114836673',
  'source_url': 'http://www.vat-lookup.co.uk/verify/vat_check.php/VATNumber/GB114836673',
  'query': '"24 SEVEN SAMEDAY LTD" VAT'},
 {'candidate_vat': 'GB729317226',
  'source_url': 'http://www.vat-lookup.co.uk/verify/vat_check.php/VATNumber/GB349219579/CompanyName/FLEX+FLEET+SERVICES+LTD',
  'query': '"24 SEVEN SAMEDAY LTD" VAT'},
 {'candidate_vat': 'GB770301267',
  'source_url': 'http://www.vat-lookup.co.uk/verify/vat_che

In [55]:
def discover_vat_candidates(company_name, company_number):

    queries = [
        f'"{company_name}" "VAT number"',
        f'"{company_name}" "VAT registration"',
        f'"{company_name}" VAT',
        f'"{company_name}" "{company_number}" VAT'
    ]

    candidates = []

    for query in queries:
        try:
            data = serper_search(query)

            for result in data.get("organic", []):

                title = result.get("title", "")
                snippet = result.get("snippet", "")
                link = result.get("link", "")

                search_text = title + " " + snippet

                vats = extract_vat_candidates(search_text)

                for vat in vats:
                    candidates.append({
                        "candidate_vat": vat,
                        "source_url": link,
                        "title": title,
                        "snippet": snippet,
                        "query": query
                    })

        except Exception as e:
            print("Search error:", company_name, e)

    return candidates

In [56]:
test_candidates = discover_vat_candidates(
    "24 SEVEN SAMEDAY LTD",
    "03793935"
)

pd.DataFrame(test_candidates)[
    ["candidate_vat", "title", "snippet", "source_url"]
]

,candidate_vat,title,snippet,source_url
0,GB770300368,24 SEVEN SAMEDAY LTD VAT Registration Informat...,Company name on VAT Registration: 24 SEVEN SAM...,http://vat-lookup.com/verify/vat_check.php/VAT...
1,GB770300368,MILES PLATTS LTD VAT Registration Information ...,... VAT number. Research company registration ...,http://www.vat-lookup.co.uk/verify/vat_check.p...
2,GB770300368,Value Added Tax information for FLEX FLEET SER...,... VAT number. Research company registration ...,http://www.vat-lookup.co.uk/verify/vat_check.p...
3,GB496785415,David Michael Hancock — Director Profile,"24 SEVEN SAMEDAY LTD. director · Transport, Ha...",https://www.formationdata.co.uk/director/david...
4,GB740669225,IADA LTD VAT Registration Information for GB74...,... VAT number. Research company registration ...,http://www.vat-lookup.co.uk/verify/vat_check.p...
5,GB770300368,IADA LTD VAT Registration Information for GB74...,... VAT number. Research company registration ...,http://www.vat-lookup.co.uk/verify/vat_check.p...
6,GB770300368,24 SEVEN SAMEDAY LTD VAT Registration Informat...,Company name on VAT Registration: 24 SEVEN SAM...,http://vat-lookup.com/verify/vat_check.php/VAT...
7,GB770300368,Value Added Tax information for FLEX FLEET SER...,Company name on VAT Registration: FLEX FLEET S...,http://www.vat-lookup.co.uk/verify/vat_check.p...
8,GB770300368,MILES PLATTS LTD VAT Registration Information ...,Company name on VAT Registration: MILES PLATTS...,http://www.vat-lookup.co.uk/verify/vat_check.p...
9,GB740669225,IADA LTD VAT Registration Information for GB74...,Company name on VAT Registration: IADA LTD. VA...,http://www.vat-lookup.co.uk/verify/vat_check.p...


In [57]:
def normalize_company_name(name):
    name = str(name).upper()

    # eliminăm punctuația
    name = re.sub(r"[^A-Z0-9\s]", " ", name)

    # eliminăm spațiile multiple
    name = re.sub(r"\s+", " ", name).strip()

    return name


def filter_candidates_by_company(candidates, company_name):

    target = normalize_company_name(company_name)

    accepted = []

    for item in candidates:

        title = normalize_company_name(
            item.get("title", "")
        )

        snippet = normalize_company_name(
            item.get("snippet", "")
        )

        # numele trebuie să apară în titlu sau snippet
        if target in title or target in snippet:
            accepted.append(item)

    return accepted

In [58]:
filtered_candidates = filter_candidates_by_company(
    test_candidates,
    "24 SEVEN SAMEDAY LTD"
)

pd.DataFrame(filtered_candidates)[
    [
        "candidate_vat",
        "title",
        "snippet",
        "source_url"
    ]
]

,candidate_vat,title,snippet,source_url
0,GB770300368,24 SEVEN SAMEDAY LTD VAT Registration Informat...,Company name on VAT Registration: 24 SEVEN SAM...,http://vat-lookup.com/verify/vat_check.php/VAT...
1,GB770300368,MILES PLATTS LTD VAT Registration Information ...,... VAT number. Research company registration ...,http://www.vat-lookup.co.uk/verify/vat_check.p...
2,GB770300368,Value Added Tax information for FLEX FLEET SER...,... VAT number. Research company registration ...,http://www.vat-lookup.co.uk/verify/vat_check.p...
3,GB496785415,David Michael Hancock — Director Profile,"24 SEVEN SAMEDAY LTD. director · Transport, Ha...",https://www.formationdata.co.uk/director/david...
4,GB740669225,IADA LTD VAT Registration Information for GB74...,... VAT number. Research company registration ...,http://www.vat-lookup.co.uk/verify/vat_check.p...
5,GB770300368,IADA LTD VAT Registration Information for GB74...,... VAT number. Research company registration ...,http://www.vat-lookup.co.uk/verify/vat_check.p...
6,GB770300368,24 SEVEN SAMEDAY LTD VAT Registration Informat...,Company name on VAT Registration: 24 SEVEN SAM...,http://vat-lookup.com/verify/vat_check.php/VAT...
7,GB770300368,Value Added Tax information for FLEX FLEET SER...,Company name on VAT Registration: FLEX FLEET S...,http://www.vat-lookup.co.uk/verify/vat_check.p...
8,GB770300368,MILES PLATTS LTD VAT Registration Information ...,Company name on VAT Registration: MILES PLATTS...,http://www.vat-lookup.co.uk/verify/vat_check.p...
9,GB740669225,IADA LTD VAT Registration Information for GB74...,Company name on VAT Registration: IADA LTD. VA...,http://www.vat-lookup.co.uk/verify/vat_check.p...


In [59]:
def filter_candidates_strict(candidates, company_name):

    target = normalize_company_name(company_name)

    accepted = []

    for item in candidates:

        title = normalize_company_name(
            item.get("title", "")
        )

        # acceptăm doar dacă numele companiei
        # apare în titlul rezultatului
        if target in title:
            accepted.append(item)

    return accepted

In [60]:
strict_candidates = filter_candidates_strict(
    test_candidates,
    "24 SEVEN SAMEDAY LTD"
)

pd.DataFrame(strict_candidates)[
    [
        "candidate_vat",
        "title",
        "source_url"
    ]
]

,candidate_vat,title,source_url
0,GB770300368,24 SEVEN SAMEDAY LTD VAT Registration Informat...,http://vat-lookup.com/verify/vat_check.php/VAT...
1,GB770300368,24 SEVEN SAMEDAY LTD VAT Registration Informat...,http://vat-lookup.com/verify/vat_check.php/VAT...
2,GB770300368,24 SEVEN SAMEDAY LTD VAT Registration Informat...,http://vat-lookup.com/verify/vat_check.php/VAT...


In [61]:
strict_df = pd.DataFrame(strict_candidates)

if not strict_df.empty:
    unique_candidates = (
        strict_df
        .drop_duplicates(subset=["candidate_vat"])
        .reset_index(drop=True)
    )

    display(
        unique_candidates[
            ["candidate_vat", "title", "source_url"]
        ]
    )
else:
    print("No candidates passed the strict filter.")

,candidate_vat,title,source_url
0,GB770300368,24 SEVEN SAMEDAY LTD VAT Registration Informat...,http://vat-lookup.com/verify/vat_check.php/VAT...


In [64]:
def discover_company_vats(company_name, company_number):

    raw_candidates = discover_vat_candidates(
        company_name,
        company_number
    )

    filtered = filter_candidates_strict(
        raw_candidates,
        company_name
    )

    # deduplicare după VAT
    unique = {}

    for item in filtered:
        vat = item["candidate_vat"]

        if vat not in unique:
            unique[vat] = item

    return list(unique.values())

In [65]:
import time

discovery_results = []

companies_to_process = final_results[
    final_results["final_status"] == "NOT_PROCESSED"
]

total = len(companies_to_process)

for i, (_, row) in enumerate(companies_to_process.iterrows(), start=1):

    company_name = row["CompanyName"]
    company_number = row["CompanyNumber"]

    print(f"[{i}/{total}] {company_name}")

    try:
        candidates = discover_company_vats(
            company_name,
            company_number
        )

        if candidates:

            for candidate in candidates:
                discovery_results.append({
                    "CompanyName": company_name,
                    "CompanyNumber": company_number,
                    "candidate_vat": candidate["candidate_vat"],
                    "source_url": candidate["source_url"],
                    "title": candidate["title"],
                    "query": candidate["query"],
                    "discovery_status": "CANDIDATE_FOUND"
                })

        else:
            discovery_results.append({
                "CompanyName": company_name,
                "CompanyNumber": company_number,
                "candidate_vat": None,
                "source_url": None,
                "title": None,
                "query": None,
                "discovery_status": "NO_CANDIDATE_FOUND"
            })

    except Exception as e:

        discovery_results.append({
            "CompanyName": company_name,
            "CompanyNumber": company_number,
            "candidate_vat": None,
            "source_url": None,
            "title": None,
            "query": None,
            "discovery_status": "SEARCH_FAILED"
        })

        print("ERROR:", e)

    # salvăm progresul la fiecare 25 companii
    if i % 25 == 0:
        pd.DataFrame(discovery_results).to_csv(
            "discovery_progress.csv",
            index=False
        )

    # mică pauză între companii
    time.sleep(0.2)

[1/499] ATM POINT LTD
[2/499] A2E VENTURE CATALYSTS LIMITED
[3/499] BEDFORD INSURANCE SERVICES GROUP LIMITED
[4/499] BRANKSOME LEISURE LIMITED
[5/499] ASR INVESTMENT SOLUTIONS LTD
[6/499] ABHM LIMITED
[7/499] BARRIERS AND GATES DIRECT LIMITED
[8/499] BRENSCOMBE MANAGEMENT COMPANY LIMITED
[9/499] BRAITHWAITE INDUSTRIES LTD
[10/499] AUNTIE JANIS'S LTD
[11/499] ANK BRISTOL LIMITED
[12/499] ALBRIGHT ELECTRICAL SOLUTIONS LTD
[13/499] B.S.B. (NORMANTON) LIMITED
[14/499] BIRMINGHAM CENTRAL AESTHETICS LTD
[15/499] AUDBY HOUSE LIMITED
[16/499] BRAND CONNECT PROMOTIONS LTD
[17/499] ANSTEY MANOR SCHOOL LIMITED
[18/499] 10FY LTD
[19/499] AJ TECH CONSULTING SERVICES LTD
[20/499] AP ATHLETICS LTD
[21/499] AGON SYSTEMS LIMITED
[22/499] 23VS PROPERTIES LTD
[23/499] AFC CHORLTON LTD
[24/499] AL AREZ SOUTH KEN TRADING LTD
[25/499] 47 NEVERN SQUARE LIMITED
[26/499] ABCD GROUP LIMITED
[27/499] BEAGLE.AI LTD
[28/499] ARUN TECHNOLOGY LTD
[29/499] ADVANCED MEDICAL SOLUTIONS LIMITED
[30/499] AIR STORE U.K. LT

In [66]:
discovery_df = pd.DataFrame(discovery_results)

discovery_df.to_csv(
    "discovery_results_499.csv",
    index=False
)

In [67]:
discovery_df["discovery_status"].value_counts()

,count
discovery_status,
NO_CANDIDATE_FOUND,482
CANDIDATE_FOUND,20


In [68]:
print(
    "Unique companies with candidates:",
    discovery_df.loc[
        discovery_df["discovery_status"] == "CANDIDATE_FOUND",
        "CompanyNumber"
    ].nunique()
)

print(
    "Unique VAT candidates:",
    discovery_df["candidate_vat"].dropna().nunique()
)

Unique companies with candidates: 17
Unique VAT candidates: 6


In [69]:
candidates_df = discovery_df[
    discovery_df["discovery_status"] == "CANDIDATE_FOUND"
].copy()

candidates_df[
    [
        "CompanyName",
        "CompanyNumber",
        "candidate_vat",
        "title",
        "source_url"
    ]
]

,CompanyName,CompanyNumber,candidate_vat,title,source_url
14,AUDBY HOUSE LIMITED,13505320,GB389138453,Audby House Limited - Company Profile,https://open.endole.co.uk/insight/company/1350...
15,AUDBY HOUSE LIMITED,13505320,GB496785415,"Audby House Limited (13505320): Directors, Own...",https://www.formationdata.co.uk/company/audby-...
26,ABCD GROUP LIMITED,10502312,GB496785415,"Abcd Group Limited (10502312): Directors, Owne...",https://www.formationdata.co.uk/company/abcd-g...
28,ARUN TECHNOLOGY LTD,05616679,GB496785415,"Arun Technology Ltd (05616679): Directors, Own...",https://www.formationdata.co.uk/company/arun-t...
29,ADVANCED MEDICAL SOLUTIONS LIMITED,02666957,GB636555127,Advanced Medical Solutions Limited,https://www.lohmann-rauscher.com/fileadmin/upl...
42,BINGLEY REAL ESTATE LIMITED,13998817,GB496785415,Bingley Real Estate Limited (13998817),https://www.formationdata.co.uk/company/bingle...
49,ADVANCE VEHICLE RENTAL LIMITED,04661494,GB496785415,Advance Vehicle Rental Limited (04661494),https://www.formationdata.co.uk/company/advanc...
63,AIR HOST AND CLEAN LTD,11361466,GB496785415,AIR Host And Clean Ltd (11361466),https://www.formationdata.co.uk/company/air-ho...
69,BROTHERS MACHINE TOOLS LIMITED,12485367,GB496785415,Brothers Machine Tools Limited (12485367): Dir...,https://www.formationdata.co.uk/company/brothe...
70,ALBAN ENTERPRISES LIMITED,NI028710,GB496785415,Alban Enterprises Limited (NI028710): Director...,https://www.formationdata.co.uk/company/alban-...


In [70]:
vat_company_check = (
    candidates_df
    .groupby("candidate_vat")
    .agg(
        n_companies=("CompanyNumber", "nunique"),
        companies=("CompanyName", lambda x: list(set(x)))
    )
    .reset_index()
)

vat_company_check

,candidate_vat,n_companies,companies
0,GB154407423,1,[AMBASSADOR REPAIR CENTRE LIMITED]
1,GB301267054,1,[ASHBOURNE BUILDING CONSULTANCY LTD]
2,GB389138453,1,[AUDBY HOUSE LIMITED]
3,GB485462068,1,[1066 KITCHEN DESIGNS LIMITED]
4,GB496785415,15,"[BINGLEY REAL ESTATE LIMITED, ATHENA COSTS LTD..."
5,GB636555127,1,[ADVANCED MEDICAL SOLUTIONS LIMITED]


In [71]:
final_results[
    final_results["CompanyName"].str.contains(
        "AUDEBY",
        case=False,
        na=False
    )
][
    ["CompanyName", "CompanyNumber", "PostCode"]
]

,CompanyName,CompanyNumber,PostCode


In [72]:
mask = (
    (candidates_df["CompanyName"] == "ASHBOURNE BUILDING CONSULTANCY LTD") &
    (candidates_df["candidate_vat"] == "GB301267054")
)

candidates_df.loc[mask, "hmrc_status"] = "VALID"
candidates_df.loc[mask, "hmrc_name"] = "ASHBOURNE BUILDING CONSULTANCY LTD"
candidates_df.loc[mask, "entity_match"] = "MATCH"
candidates_df.loc[mask, "final_status"] = "VERIFIED"

In [73]:
mask = (
    (candidates_df["CompanyName"] == "1066 KITCHEN DESIGNS LIMITED") &
    (candidates_df["candidate_vat"] == "GB485462068")
)

candidates_df.loc[mask, "hmrc_status"] = "VALID"
candidates_df.loc[mask, "hmrc_name"] = "1066 KITCHEN DESIGNS LIMITED"
candidates_df.loc[mask, "entity_match"] = "MATCH"
candidates_df.loc[mask, "final_status"] = "VERIFIED"

In [74]:
mask = candidates_df["candidate_vat"] == "GB154407423"

candidates_df.loc[mask, "hmrc_status"] = "VALID"
candidates_df.loc[mask, "hmrc_name"] = "NCA MOTORS LIMITED"
candidates_df.loc[mask, "entity_match"] = "MISMATCH"
candidates_df.loc[mask, "final_status"] = "REJECTED_ENTITY_MISMATCH"

In [75]:
mask = candidates_df["candidate_vat"] == "GB636555127"

candidates_df.loc[mask, "hmrc_status"] = "VALID"
candidates_df.loc[mask, "hmrc_name"] = "ADVANCED MEDICAL SOLUTIONS GROUP PLC"
candidates_df.loc[mask, "entity_match"] = "MISMATCH"
candidates_df.loc[mask, "final_status"] = "REJECTED_ENTITY_MISMATCH"

In [76]:
mask = candidates_df["candidate_vat"] == "GB496785415"

candidates_df.loc[mask, "hmrc_status"] = "VALID"
candidates_df.loc[mask, "hmrc_name"] = "FORMATIONDATA SYSTEMS LTD"
candidates_df.loc[mask, "entity_match"] = "MISMATCH"
candidates_df.loc[mask, "final_status"] = "REJECTED_ENTITY_MISMATCH"

In [78]:
candidates_df[
    candidates_df["candidate_vat"] == "GB389138453"
][
    ["CompanyName", "CompanyNumber", "candidate_vat"]
]

,CompanyName,CompanyNumber,candidate_vat
14,AUDBY HOUSE LIMITED,13505320,GB389138453


In [79]:
company_number = candidates_df.loc[
    candidates_df["candidate_vat"] == "GB389138453",
    "CompanyNumber"
].iloc[0]

final_results[
    final_results["CompanyNumber"] == company_number
][
    ["CompanyName", "CompanyNumber", "PostCode"]
]

,CompanyName,CompanyNumber,PostCode
15,AUDBY HOUSE LIMITED,13505320,DE1 1TJ


In [80]:
mask = candidates_df["candidate_vat"] == "GB389138453"

candidates_df.loc[mask, "hmrc_status"] = "VALID"
candidates_df.loc[mask, "hmrc_name"] = "AUDBY HOUSE LIMITED"
candidates_df.loc[mask, "entity_match"] = "AMBIGUOUS"
candidates_df.loc[mask, "final_status"] = "REVIEW"

In [81]:
n_searched = 499

n_candidate_companies = candidates_df["CompanyNumber"].nunique()

n_verified = candidates_df.loc[
    candidates_df["final_status"] == "VERIFIED",
    "CompanyNumber"
].nunique()

n_rejected = candidates_df.loc[
    candidates_df["final_status"] == "REJECTED_ENTITY_MISMATCH",
    "CompanyNumber"
].nunique()

n_review = candidates_df.loc[
    candidates_df["final_status"] == "REVIEW",
    "CompanyNumber"
].nunique()

n_no_candidate = 482

print("Companies searched:", n_searched)
print("Companies with candidate:", n_candidate_companies)
print("Verified:", n_verified)
print("Rejected entity mismatch:", n_rejected)
print("Review / ambiguous:", n_review)
print("No candidate found:", n_no_candidate)

Companies searched: 499
Companies with candidate: 17
Verified: 2
Rejected entity mismatch: 16
Review / ambiguous: 1
No candidate found: 482


In [82]:
total_companies = 500

total_verified = n_verified + 1   # 24 SEVEN
total_rejected = n_rejected
total_review = n_review
total_no_candidate = n_no_candidate

verified_coverage = (
    total_verified / total_companies
) * 100

candidate_discovery_rate = (
    (n_candidate_companies + 1) / total_companies
) * 100

print("FINAL POC RESULTS")
print("-----------------")
print("Companies:", total_companies)
print("Verified VAT links:", total_verified)
print("Rejected entity mismatches:", total_rejected)
print("Review / ambiguous:", total_review)
print("No candidate found:", total_no_candidate)

print(
    "Verified coverage:",
    round(verified_coverage, 2),
    "%"
)

print(
    "Candidate discovery rate:",
    round(candidate_discovery_rate, 2),
    "%"
)

FINAL POC RESULTS
-----------------
Companies: 500
Verified VAT links: 3
Rejected entity mismatches: 16
Review / ambiguous: 1
No candidate found: 482
Verified coverage: 0.6 %
Candidate discovery rate: 3.6 %


In [83]:
evaluated = candidates_df[
    candidates_df["final_status"].isin([
        "VERIFIED",
        "REJECTED_ENTITY_MISMATCH"
    ])
]

correct_predictions = (
    evaluated["final_status"] == "VERIFIED"
).sum()

wrong_predictions = (
    evaluated["final_status"] == "REJECTED_ENTITY_MISMATCH"
).sum()

precision = (
    correct_predictions /
    (correct_predictions + wrong_predictions)
) * 100

false_positive_rate = (
    wrong_predictions /
    (correct_predictions + wrong_predictions)
) * 100

print("Evaluated company-VAT predictions:",
      len(evaluated))

print("Correct:", correct_predictions)
print("Wrong:", wrong_predictions)

print("Precision:", round(precision, 2), "%")
print(
    "False-positive rate:",
    round(false_positive_rate, 2),
    "%"
)

Evaluated company-VAT predictions: 19
Correct: 2
Wrong: 17
Precision: 10.53 %
False-positive rate: 89.47 %
